In [9]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "etf_adjusted_close.csv"

prices = pd.read_csv(
    DATA_PATH,
    index_col="Date",
    parse_dates=True,
)

returns = prices.pct_change(fill_method=None).dropna()

print("Prices:", prices.shape)
print("Returns:", returns.shape)
print("Missing returns:", returns.isna().sum().sum())

prices.head()

Prices: (3268, 12)
Returns: (3267, 12)
Missing returns: 0


,SPY,QQQ,IWM,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ
Date,,,,,,,,,,,,
2007-04-11,110.289658,39.279694,66.941124,52.894066,30.795134,59.255993,58.066200,62.582520,44.114380,67.080002,23.717676,45.059521
2007-04-12,110.779778,39.590656,67.391281,53.247246,31.310007,59.313496,58.079468,62.676716,44.143967,66.989998,23.857685,44.752880
2007-04-13,111.285194,39.670624,67.824745,53.423836,31.463705,59.212826,57.900051,62.570763,44.063625,67.839996,23.988363,45.258251
2007-04-16,112.341995,40.034893,68.758461,53.987541,31.837688,59.277557,58.219078,62.623730,44.046719,68.400002,23.801685,45.315048
2007-04-17,112.640648,40.123745,68.550041,53.994328,31.666058,59.529251,58.544727,62.965149,44.025558,68.000000,23.559000,45.905624


In [2]:
print("Shape:", prices.shape)
print("Date range:", prices.index.min(), "to", prices.index.max())
print("\nColumns:")
print(prices.columns.tolist())

print("\nMissing values:")
display(prices.isna().sum().sort_values(ascending=False))

print("\nFirst valid date:")
display(prices.apply(pd.Series.first_valid_index).sort_values())

Shape: (3335, 12)
Date range: 2007-01-03 00:00:00 to 2020-04-01 00:00:00

Columns:
['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'IEF', 'TLT', 'LQD', 'HYG', 'GLD', 'DBC', 'VNQ']

Missing values:


HYG    67
SPY     0
QQQ     0
IWM     0
EFA     0
EEM     0
IEF     0
TLT     0
LQD     0
GLD     0
DBC     0
VNQ     0
dtype: int64


First valid date:


SPY   2007-01-03
QQQ   2007-01-03
IWM   2007-01-03
EFA   2007-01-03
EEM   2007-01-03
IEF   2007-01-03
TLT   2007-01-03
LQD   2007-01-03
GLD   2007-01-03
DBC   2007-01-03
VNQ   2007-01-03
HYG   2007-04-11
dtype: datetime64[us]

In [3]:
quality_summary = pd.DataFrame({
    "first_date": prices.apply(pd.Series.first_valid_index),
    "last_date": prices.apply(pd.Series.last_valid_index),
    "observations": prices.notna().sum(),
    "missing": prices.isna().sum(),
    "minimum_price": prices.min(),
    "maximum_price": prices.max(),
})

print("Duplicate dates:", prices.index.duplicated().sum())
print("Non-positive prices:", (prices <= 0).sum().sum())

quality_summary

Duplicate dates: 0
Non-positive prices: 0


,first_date,last_date,observations,missing,minimum_price,maximum_price
SPY,2007-01-03,2020-04-01,3335,0,54.184429,336.362091
QQQ,2007-01-03,2020-04-01,3335,0,22.811867,236.476059
IWM,2007-01-03,2020-04-01,3335,0,29.328701,169.051758
EFA,2007-01-03,2020-04-01,3335,0,22.784172,70.661583
EEM,2007-01-03,2020-04-01,3335,0,14.375123,49.532421
IEF,2007-01-03,2020-04-01,3335,0,57.683903,121.830002
TLT,2007-01-03,2020-04-01,3335,0,55.146198,171.042877
LQD,2007-01-03,2020-04-01,3335,0,52.007923,133.932999
HYG,2007-04-11,2020-04-01,3268,67,30.295080,87.579842
GLD,2007-01-03,2020-04-01,3335,0,60.169998,184.589996


In [5]:
common_start_date = prices.apply(pd.Series.first_valid_index).max()
common_end_date = prices.apply(pd.Series.last_valid_index).min()

print("Common sample:", common_start_date, "to", common_end_date)

common_prices = prices.loc[common_start_date:common_end_date].dropna()

print("Common sample shape:", common_prices.shape)
print("Remaining missing values:", common_prices.isna().sum().sum())

Common sample: 2007-04-11 00:00:00 to 2020-04-01 00:00:00
Common sample shape: (3268, 12)
Remaining missing values: 0


In [7]:
returns = common_prices.pct_change(fill_method=None).dropna()

return_summary = returns.describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
).T

return_summary["minimum_date"] = returns.idxmin()
return_summary["maximum_date"] = returns.idxmax()

return_summary

,count,mean,std,min,1%,5%,50%,95%,99%,max,minimum_date,maximum_date
SPY,3267.0,0.000331,0.013029,-0.109424,-0.041953,-0.019380,0.000636,0.016955,0.035424,0.145198,2020-03-16,2008-10-13
QQQ,3267.0,0.000566,0.013843,-0.119788,-0.039967,-0.022436,0.001043,0.019709,0.037810,0.121647,2020-03-16,2008-10-13
IWM,3267.0,0.000269,0.015859,-0.132669,-0.046653,-0.024271,0.000995,0.022448,0.046646,0.091491,2020-03-16,2020-03-24
EFA,3267.0,0.000101,0.014916,-0.111632,-0.045644,-0.022184,0.000604,0.019501,0.041898,0.158876,2008-09-29,2008-10-13
EEM,3267.0,0.000210,0.019643,-0.161662,-0.057319,-0.028099,0.000892,0.025135,0.055446,0.227698,2008-10-15,2008-10-13
IEF,3267.0,0.000230,0.004323,-0.025073,-0.011090,-0.006717,0.000360,0.006896,0.011301,0.034262,2020-03-17,2009-03-18
TLT,3267.0,0.000368,0.009490,-0.066683,-0.022950,-0.014647,0.000670,0.014500,0.025380,0.075195,2020-03-17,2020-03-20
LQD,3267.0,0.000219,0.005660,-0.091111,-0.011668,-0.006310,0.000430,0.006277,0.011650,0.097678,2008-09-29,2008-09-30
HYG,3267.0,0.000189,0.007555,-0.080975,-0.023107,-0.008902,0.000330,0.008927,0.020117,0.122689,2008-09-29,2008-10-13
GLD,3267.0,0.000311,0.011435,-0.087808,-0.031079,-0.017837,0.000477,0.017798,0.029704,0.112905,2013-04-15,2008-09-17
